# 01 — Synthetic OHS Data Generation
---
**What this notebook does:**
This notebook generates **100 synthetic Ontario OHS (Occupational Health & Safety) inspection records**
using the Python `Faker` library with a Canadian locale (`en_CA`).

These synthetic records act as labeled training data for the two downstream ML models:
- **Model 1 (NLP):** uses `INSPECTION_TEXT_DESCRIPTION` to predict `SEVERITY_LEVEL`
- **Model 2 (ML):** uses all structured columns + severity to predict `RISK_SCORE`

**Output:** Delta table → `workspace.ohs_data.synthetic_inspections`

**Schema:** 21 original OHS columns + 3 new columns:
| New Column | Type | Purpose |
|---|---|---|
| `INSPECTION_TEXT_DESCRIPTION` | string | Free-text inspection narrative — NLP model input |
| `SEVERITY_LEVEL` | string | Critical / High / Medium / Low — Model 1 target label |
| `RISK_SCORE` | float | 0–100 continuous score — Model 2 target label |

**Catalog:** `workspace` &nbsp;|&nbsp; **Schema:** `ohs_data` &nbsp;|&nbsp; **Runtime:** Serverless

## Cell 1 — Install Dependencies
Installs the `faker` library for generating realistic synthetic Canadian data.
`--quiet` suppresses verbose pip output. Databricks restarts the Python
interpreter after any `%pip install`, so this must be the very first cell.

In [0]:
%pip install faker --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Cell 2 — Imports and Random Seed Setup
Import all required libraries and fix every random seed to **42** so that
re-running this notebook always produces the exact same 500 records.
`en_CA` locale gives us Ontario city names, postal code prefixes,
Canadian company names, and street addresses.

In [0]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime

# Canadian English locale — gives Ontario-specific addresses and company names
fake = Faker("en_CA")

# Fix all three random sources so every run is fully reproducible
Faker.seed(42)
random.seed(42)
np.random.seed(42)

## Cell 3 — OHS Domain Constants
Define all valid dropdown values that mirror the real Ontario OHS dataset
(taken from the uploaded data dictionary and Excel files).
These lists are used both for synthetic data generation AND as the
valid input options in the Flask app's form dropdowns.

In [0]:
# ── NAICS codes relevant to Ontario OHS inspections ─────────────────────────
# Each tuple is (naics_code, naics_description)
# Covers the main sectors: Construction, Industrial, Healthcare, Mining
NAICS_OPTIONS = [
    ("236110", "Residential Building Construction"),
    ("236210", "Industrial Building Construction"),
    ("237110", "Water and Sewer Line Construction"),
    ("238110", "Poured Concrete Foundation Contractors"),
    ("238210", "Electrical Contractors and Other Wiring Installation"),
    ("238310", "Drywall and Insulation Contractors"),
    ("311511", "Fluid Milk Manufacturing"),
    ("321111", "Sawmills"),
    ("331110", "Iron and Steel Mills"),
    ("332310", "Prefabricated Metal Building Manufacturing"),
    ("335930", "Wiring Device Manufacturing"),
    ("621110", "Offices of Physicians"),
    ("622110", "General Medical and Surgical Hospitals"),
    ("623110", "Nursing Care Facilities"),
    ("212110", "Bituminous Coal Underground Mining"),
    ("212220", "Gold and Silver Ore Mining"),
    ("212315", "Limestone Quarrying and Processing"),
    ("722511", "Full-Service Restaurants"),
    ("811111", "General Automotive Repair"),
    ("337110", "Wood Kitchen Cabinet Manufacturing"),
]

# ── Case and visit type fields ───────────────────────────────────────────────
FIELD_VISIT_TYPES = ["Field Visit", "Field Visit Support Role", "Offsite Visit"]
CASE_TYPES        = ["Inspection", "Investigation", "Consultation"]
CASE_STATUSES     = ["Closed", "Open", "In Progress"]
CONTRAVENER_ROLES = ["Employer", "Constructor", "Supervisor", "Owner", "Worker", "Supplier"]

# ── Order-level fields ────────────────────────────────────────────────────────
# Stop Work Order is the most severe — used to bias Critical/High severity records
ORDER_TYPES = [
    "Forthwith Order",       # comply immediately
    "Time Based Order",      # comply within a set deadline
    "Stop Work Order",       # halt all work — most severe
    "Plan Order",            # submit a corrective plan
    "Time Unknown Order",    # timeline not yet defined
    "Requirement Forthwith", # regulatory requirement, immediate
    "Requirement Time Based",# regulatory requirement, deadline-based
]
ORDER_STATUSES = [
    "Complied With",        # employer has fixed the issue
    "Not Complied With",    # employer has NOT fixed the issue — highest risk signal
    "Outstanding",          # not yet resolved
    "In-Process",           # remediation underway
    "Withdrawn/Cancelled",  # order was cancelled
    "Rescinded",            # order was formally revoked
]

# ── Legislative references ────────────────────────────────────────────────────
CASE_ACTS = [
    "Occupational Health and Safety Act",
    "Building Opportunities in the Skilled Trades Act",
]
# Each tuple is (regulation_id, regulation_full_name)
REGULATIONS = [
    ("REG_851",  "Industrial Establishments"),
    ("REG_213",  "Construction Projects"),
    ("REG_490",  "Designated Substance - Asbestos on Construction Projects"),
    ("REG_632",  "Workplace Hazardous Materials Information System"),
    ("REG_297",  "Firefighters - Protective Equipment"),
    ("REG_O67",  "Occupational Health and Safety Act - General"),
    ("REG_854",  "Mines and Mining Plants"),
    ("REG_388",  "Electrical Utility Safety Rules"),
    ("REG_420",  "Notices and Reports - Fatalities and Critical Injuries"),
]

# ── Section / subsection / clause references ─────────────────────────────────
SECTIONS    = ["25", "26", "27", "28", "29", "43", "50", "54", "55", "57"]
SUBSECTIONS = ["(1)", "(2)", "(3)", "(4)", "(5)", ""]   # empty = not applicable
CLAUSES     = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", ""]

# ── Ontario cities with their postal code first-letter prefix ─────────────────
ONTARIO_CITIES = [
    ("Toronto", "M"),        ("Mississauga", "L"),   ("Ottawa", "K"),
    ("Hamilton", "L"),       ("London", "N"),         ("Brampton", "L"),
    ("Windsor", "N"),        ("Kitchener", "N"),      ("Thunder Bay", "P"),
    ("Sudbury", "P"),        ("Barrie", "L"),         ("Oshawa", "L"),
    ("Markham", "L"),        ("Vaughan", "L"),         ("Oakville", "L"),
    ("Guelph", "N"),         ("Cambridge", "N"),       ("Burlington", "L"),
    ("St. Catharines", "L"), ("Peterborough", "K"),
]

## Cell 4 — Labeled Inspection Text Templates

This is the most important cell for Model 1 training.

**How it works:**
Each severity tier (`Critical`, `High`, `Medium`, `Low`) has a pool of
realistic OHS inspection narratives written to match that severity level.
When generating a record, we:
1. Randomly draw a severity tier (weighted: more Medium/Low than Critical)
2. Randomly pick a text from that tier's pool
3. This creates a **labeled** corpus — the text IS the ground truth for the NLP model

In a real deployment, these labels would come from human inspectors.
For this POC, the template pools simulate that labelling process.

In [0]:
# Text templates keyed by severity level
# Each template is a realistic Inspector's narrative describing findings
# Critical = imminent danger, fatalities, Stop Work Orders
# High     = serious violations, significant injury risk
# Medium   = moderate violations, corrective action required
# Low      = minor administrative or housekeeping issues
SEVERITY_TEMPLATES = {
    "Critical": [
        "Inspector attended following a fatal fall from height. Worker found at the base of an unguarded scaffold approximately 8 metres high. No fall protection equipment was present on site. Site immediately shut down under Stop Work Order. Evidence of repeated non-compliance with fall arrest regulations. Multiple workers observed without any PPE.",
        "Report received of chemical explosion in industrial facility. Three workers sustained critical burns requiring immediate hospitalization. Improper storage of flammable materials adjacent to ignition sources confirmed. WHMIS labels absent from all chemical containers. Emergency response plan non-existent. Critical violation of O. Reg. 851.",
        "Worker critically injured when caught in unguarded rotating machinery. Lockout tagout procedures were not followed. Machine guarding was removed without replacement. Supervisor was aware of the hazard and failed to act. Immediate Stop Work Order issued pending full fatality investigation.",
        "Workers found confined in unshored trench over 1.5 metres deep with visible soil instability and cracking. One worker had been partially buried and required emergency rescue. Constructor had no written excavation safety plan. Stop Work Order issued immediately for all excavation activities.",
        "Investigation confirmed fatal electrocution of apprentice electrician during panel work. Energized conductors were not properly isolated. Lockout tagout procedures were absent from the workplace entirely. No qualified supervisor present on site. Employer had prior knowledge of unsafe electrical conditions.",
        "Worker fatality confirmed from fall through unguarded roof opening. Opening had no covers, guardrails, or safety barriers. No elevated work plan available at site. Constructor had been advised of this opening three days prior by another worker. Full investigation launched under OHSA section 51.",
    ],
    "High": [
        "Workers found operating forklifts without valid operator certification. Multiple near-miss incidents reported in the previous 30 days. Pedestrian zones not delineated and aisle markings absent. Inadequate lighting in storage areas creating collision risk. Safe work procedures for powered industrial trucks not available at workplace.",
        "Asbestos-containing materials identified in active renovation area. Workers not informed of asbestos presence prior to beginning work. No Type 3 asbestos work plan submitted to Ministry. Workers not wearing respiratory protection. Hygienist ordered to conduct air quality monitoring. Immediate remediation required under O. Reg. 278.",
        "Nurses reporting musculoskeletal injuries from manual patient handling without mechanical lift aids. Safe patient handling policy not posted in facility. Staff insufficiently trained in manual transfer techniques. Healthcare facility ordered to implement a mechanical patient handling program within 30 days.",
        "Electrical panel found with exposed live conductors accessible to general workers. No lockout procedure posted at the panel. Workers performing nearby maintenance without de-energizing equipment. Last qualified electrical inspection was over 5 years ago. Time Based Order issued for immediate qualified electrician review.",
        "Multiple fall hazards identified on multi-storey construction site. Floor openings unguarded with no covers or safety nets. Ladders improperly secured and visibly damaged. Two workers observed at height without personal fall arrest systems engaged. Site safety coordinator absent at time of inspection.",
        "Silica dust exposure levels exceeding occupational exposure limits found in tile cutting operation. Workers performing dry cutting without respiratory protection of any type. No exposure control plan for crystalline silica in place. Air sampling required. Medical surveillance program absent for long-term workers.",
    ],
    "Medium": [
        "Several missing safety data sheets for chemicals used on site. Workers were unable to identify hazards associated with cleaning products used daily. WHMIS training records incomplete for 4 of 12 workers. Compliance order issued requiring updated training and documentation within 30 days.",
        "Construction site showed inadequate housekeeping throughout work areas. Tripping hazards from lumber and scrap material in walkways. First aid kit missing several required supplies including bandages and eye wash solution. No evidence of daily site supervisor inspections. Corrective action required within 14 days.",
        "Healthcare facility lacks adequate sharps disposal containers in three patient care areas. Biological waste not properly segregated from general waste in those rooms. One worker reported a needlestick injury in the past month. Corrective action and updated training required within 21 days.",
        "Restaurant kitchen workers handling hot liquids without appropriate heat-resistant PPE. No written safe work procedure for handling cooking with hot oil. One thermal burn injury reported to management in past quarter. Compliance order requiring written procedure and training update issued.",
        "Workers using respiratory equipment without required fit testing completed within required periods. Respirator maintenance records absent. Hazardous dust levels measured near grinding operations exceed guidelines. Ventilation system inspection overdue by more than two years. Time-based order issued for compliance.",
        "Construction roadway workers missing required traffic control measures. Flaggers not wearing high-visibility vests and traffic protection plan not posted at site. Workers at risk of vehicle contact. Employer corrected vest issue immediately but traffic plan must be submitted and approved within 10 days.",
    ],
    "Low": [
        "Routine proactive inspection identified emergency exit signage requiring replacement due to significant fading. Two signs were not illuminated as required by regulation. Exit pathways were clear and unobstructed throughout. Corrective action required within 30 days. Employer acknowledged and agreed to repair promptly.",
        "Minor housekeeping issues observed in general storage area. One shelving unit slightly overloaded in a single location. No immediate safety risk identified at time of visit. Verbal direction given to employer to address overloading within 5 days. Employer was cooperative and began sorting items during inspection.",
        "Monthly safety meeting records not maintained for the previous 2 months. Joint health and safety committee meeting minutes not posted on the required bulletin board. No worker complaints received in this period. Administrative forthwith requirement issued to restore documentation practice. No injury history associated.",
        "Fire extinguisher inspection tags were 2 months overdue on 3 units in the facility. Extinguishers were in good physical condition with no visible damage or low pressure indicators. Forthwith order issued. Employer agreed to schedule a certified technician inspection on the same day of visit.",
        "Personal protective equipment storage area lacked proper labelling for item types and sizes. Some PPE items stored in proximity to chemical storage without adequate separation. Minor administrative violation with no worker exposure incidents on record. Written direction provided to the employer with 21-day compliance deadline.",
        "Employer's health and safety policy was three years out of date and did not reference current regulatory requirements. No recent injuries on record at this site. Owner was cooperative and began policy review during inspection. Time-based order to update and repost policy within 21 days.",
    ],
}

## Cell 5 — Helper Functions
Three utility functions used during record generation:
- `random_postal_code()` — generates a valid Ontario-format postal code (e.g. `L4T 2A9`)
- `get_sector()` — derives the industry sector from a NAICS code prefix
- `compute_risk_score()` — calculates a pseudo ground-truth risk score using
  domain business rules (this simulates what a human analyst would assign)

In [0]:
def random_postal_code(prefix: str = "L") -> str:
    """
    Generate a realistic Ontario-format postal code.
    Format: [Letter][Digit][Letter] [Digit][Letter][Digit]
    Example: L4T 2A9
    The prefix letter determines the city region (M=Toronto, L=GTA, N=SW Ontario, etc.)
    """
    d1 = random.randint(0, 9)
    a1 = random.choice("ABCEGHJKLMNPRSTVWXYZ")  # valid Canadian postal code letters
    d2 = random.randint(0, 9)
    a2 = random.choice("ABCEGHJKLMNPRSTVWXYZ")
    d3 = random.randint(0, 9)
    return f"{prefix}{d1}{a1} {d2}{a2}{d3}"


def get_sector(naics_code: str) -> str:
    """
    Derive the broad industry sector from the first 2-3 digits of the NAICS code.
    Used as a feature in Model 2 (certain sectors have higher baseline risk).
    Construction and Mining carry the highest structural risk in OHS data.
    """
    n = str(naics_code)
    if n.startswith("23"): return "Construction"   # NAICS 23xxxx = Construction
    if n.startswith("21"): return "Mining"          # NAICS 21xxxx = Mining
    if n.startswith("6"):  return "Health Care"     # NAICS 6xxxxx = Health Care
    return "Industrial"                             # Everything else = Industrial


def compute_risk_score(
    severity: str,
    order_type: str,
    order_status: str,
    case_type: str,
    naics_code: str,
) -> float:
    """
    Compute a pseudo ground-truth risk score (0–100) from domain business rules.

    This function acts as a surrogate for human analyst risk labelling.
    In a real deployment, this score would come from historical outcome data
    (e.g. repeat violations, subsequent injuries, time to compliance).

    Logic:
      base score    = driven by severity tier (Critical=82, Low=16)
      + order adj   = Stop Work Orders add +12 pts; compliant orders subtract pts
      + status adj  = Not Complied With adds +14 pts; Complied With subtracts pts
      + case adj    = Investigations (reactive) add +6 pts
      + sector adj  = Mining (+9) and Construction (+6) carry structural risk premium
      + noise       = Gaussian noise (mean=0, std=7) for natural variation
    """
    # Base score anchored to severity tier
    base = {"Critical": 82.0, "High": 63.0, "Medium": 38.0, "Low": 16.0}[severity]

    # Add Gaussian noise to simulate natural variation between similar incidents
    noise = random.gauss(0, 7)

    # Order type adjustment — Stop Work Orders indicate the most dangerous conditions
    order_adj = {
        "Stop Work Order": 12,       "Plan Order": 6,
        "Time Unknown Order": 8,     "Time Based Order": 3,
        "Forthwith Order": -2,       "Requirement Time Based": 1,
        "Requirement Forthwith": -3,
    }.get(order_type, 0)

    # Compliance status adjustment — non-compliance significantly elevates risk
    status_adj = {
        "Not Complied With": 14,     "Outstanding": 9,
        "In-Process": 2,             "Complied With": -8,
        "Withdrawn/Cancelled": -5,   "Rescinded": -3,
    }.get(order_status, 0)

    # Case type adjustment — Investigations are reactive (incident already occurred)
    case_adj = {"Investigation": 6, "Inspection": 0, "Consultation": -5}.get(case_type, 0)

    # Sector risk premium — Mining and Construction are structurally higher risk
    sector_adj = {
        "Construction": 6, "Mining": 9, "Health Care": 4, "Industrial": 2,
    }.get(get_sector(naics_code), 0)

    # Clamp final score to valid range [0, 100] and round to 1 decimal place
    return round(float(min(100.0, max(0.0, base + noise + order_adj + status_adj + case_adj + sector_adj))), 1)

## Cell 6 — Record Generator Function

`generate_ohs_records(n)` builds `n` complete OHS inspection records.

**Key design decisions:**
- Severity is sampled with **weighted probabilities** (10% Critical, 25% High, 40% Medium, 25% Low)
  to reflect the real-world distribution where most inspections are routine
- **Domain biases** are applied to make the data realistic:
  - Critical/High severity → 40% chance of Stop Work Order
  - Critical/High severity → 50% chance of Investigation case type
  - Low severity → Stop Work Orders replaced with minor order types
- Risk scores are computed from the same record's fields, creating
  **internally consistent correlations** between features and target

In [0]:
def generate_ohs_records(n: int = 100) -> pd.DataFrame:
    """
    Generate n synthetic OHS inspection records.
    Returns a pandas DataFrame with 24 columns (21 original + 3 new).
    """
    records = []

    for _ in range(n):

        # ── Step 1: Draw severity tier first — this drives everything else ───
        severity = random.choices(
            ["Critical", "High", "Medium", "Low"],
            weights=[10, 25, 40, 25],   # realistic field distribution
        )[0]

        # Pick a realistic inspection narrative from that severity's text pool
        text = random.choice(SEVERITY_TEMPLATES[severity])

        # ── Step 2: Generate visit and workplace details ──────────────────────
        visit_date             = fake.date_between(start_date="-2y", end_date="today")
        naics_code, naics_desc = random.choice(NAICS_OPTIONS)
        city, postal_prefix    = random.choice(ONTARIO_CITIES)
        postal                 = random_postal_code(postal_prefix)
        address                = f"{fake.street_address()}, {city}, ON"

        # ── Step 3: Apply domain biases to order type ─────────────────────────
        order_type = random.choice(ORDER_TYPES)

        # Severe incidents are more likely to result in a Stop Work Order
        if severity in ("Critical", "High") and random.random() < 0.4:
            order_type = "Stop Work Order"

        # Low-severity incidents should not realistically receive a Stop Work Order
        elif severity == "Low" and order_type == "Stop Work Order":
            order_type = random.choice(["Forthwith Order", "Requirement Forthwith"])

        order_status = random.choice(ORDER_STATUSES)

        # ── Step 4: Apply domain biases to case type ──────────────────────────
        case_type = random.choice(CASE_TYPES)

        # Critical/High incidents are more likely to be reactive Investigations
        if severity in ("Critical", "High") and random.random() < 0.5:
            case_type = "Investigation"

        # ── Step 5: Pick remaining fields ─────────────────────────────────────
        case_act         = random.choice(CASE_ACTS)
        reg_id, reg_name = random.choice(REGULATIONS)
        section          = random.choice(SECTIONS)
        subsection       = random.choice(SUBSECTIONS)
        clause           = random.choice(CLAUSES)

        # ── Step 6: Compute target labels ─────────────────────────────────────
        # SEVERITY_LEVEL is already determined (used as Model 1 target)
        # RISK_SCORE is derived from the record's own fields (Model 2 target)
        risk_score = compute_risk_score(
            severity, order_type, order_status, case_type, naics_code
        )

        # ── Step 7: Assemble the record dict ──────────────────────────────────
        records.append({
            # Original OHS dataset columns
            "FIELD_VISIT_DATE":             visit_date.strftime("%Y-%m-%d"),
            "FIELD_VISIT_TYPE":             random.choice(FIELD_VISIT_TYPES),
            "CASE_TYPE":                    case_type,
            "CASE_STATUS":                  random.choice(CASE_STATUSES),
            "WORKPLACE_ID":                 f"WP{fake.numerify('########')}",
            "WORKPLACE_NAME_AT_FV_TIME":    fake.company(),
            "WORKPLACE_ADDRESS_AT_FV_TIME": address,
            "POSTAL_CODE":                  postal,
            "PRIMARY_NAICS":               naics_code,
            "NAICS_DESCRIPTION":           naics_desc,
            "CONTRAVENER_ROLE":            random.choice(CONTRAVENER_ROLES),
            "CONTRAVENER_ORG_ID":          f"ORG{fake.numerify('######')}",
            # 30% chance contravener is an individual, 70% an organization
            "CONTRAVENER_NAME":            fake.company() if random.random() > 0.3 else fake.name(),
            "ORDER_TYPE":                  order_type,
            "ORDER_STATUS":                order_status,
            "CASE_ACT":                    case_act,
            "ACT_REG_ID":                  reg_id,
            "ACT_REGULATION_NAME":         reg_name,
            "SEC":                         section,
            "SUBSEC":                      subsection,
            "CLAUSE":                      clause,
            # New columns added for this project
            "INSPECTION_TEXT_DESCRIPTION": text,        # NLP model input
            "SEVERITY_LEVEL":              severity,    # Model 1 target label
            "RISK_SCORE":                  risk_score,  # Model 2 target label
        })

    return pd.DataFrame(records)

## Cell 7 — Generate Data, Display Sample, and Save to Delta

- Generates 500 records (large enough to train both models)
- Displays the first 20 rows using Databricks `display()` for interactive exploration
- Prints severity and risk score distributions so you can verify the data looks realistic
- Saves the full dataset as a **Delta table** in Unity Catalog

**After this cell runs, verify in the Catalog sidebar:**
`workspace → ohs_data → synthetic_inspections` should appear with 500 rows

In [0]:
# Generate the full synthetic dataset
df = generate_ohs_records(100)

# ── Display sample of 20 records in Databricks table viewer ──────────────────
print("=" * 60)
print("SAMPLE — FIRST 20 RECORDS")
print("=" * 60)
display(df.head(20))

# ── Print distribution summaries to verify realistic proportions ─────────────
print(f"\nTotal records generated : {len(df)}")

print("\nSeverity distribution (expected ≈ 10% Critical, 25% High, 40% Medium, 25% Low):")
print(df["SEVERITY_LEVEL"].value_counts())

print("\nRisk score statistics (expected range: 0–100, mean ≈ 45):")
print(df["RISK_SCORE"].describe().round(2))

print("\nOrder type distribution:")
print(df["ORDER_TYPE"].value_counts())

# ── Persist to Unity Catalog as a Delta table ─────────────────────────────────
# NOTE: Using the 'workspace' catalog (confirmed from your Databricks sidebar).
# If you see a catalog error, go to the Catalog icon in the left sidebar and
# confirm the top-level catalog name — replace 'workspace' if it differs.
spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS ohs_data")   # creates the schema if it doesn't exist yet

# Convert pandas DataFrame to Spark DataFrame for Delta write
spark_df = spark.createDataFrame(df)

# Write as Delta table — overwrite mode so re-running the notebook is safe
(spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")       # allows schema changes on re-run
    .saveAsTable("workspace.ohs_data.synthetic_inspections"))

print("\n✓  Saved → workspace.ohs_data.synthetic_inspections")
print("    Go to Catalog → workspace → ohs_data → synthetic_inspections to verify.")

SAMPLE — FIRST 20 RECORDS


FIELD_VISIT_DATE,FIELD_VISIT_TYPE,CASE_TYPE,CASE_STATUS,WORKPLACE_ID,WORKPLACE_NAME_AT_FV_TIME,WORKPLACE_ADDRESS_AT_FV_TIME,POSTAL_CODE,PRIMARY_NAICS,NAICS_DESCRIPTION,CONTRAVENER_ROLE,CONTRAVENER_ORG_ID,CONTRAVENER_NAME,ORDER_TYPE,ORDER_STATUS,CASE_ACT,ACT_REG_ID,ACT_REGULATION_NAME,SEC,SUBSEC,CLAUSE,INSPECTION_TEXT_DESCRIPTION,SEVERITY_LEVEL,RISK_SCORE
2025-09-09,Field Visit,Inspection,Open,WP81960013,Wagner Inc,"433 Jill Springs, Burlington, ON",L3X 6K7,236110,Residential Building Construction,Supervisor,ORG838637,"Ramirez, Booth and Blake",Stop Work Order,Complied With,Building Opportunities in the Skilled Trades Act,REG_297,Firefighters - Protective Equipment,27,(2),,Multiple fall hazards identified on multi-storey construction site. Floor openings unguarded with no covers or safety nets. Ladders improperly secured and visibly damaged. Two workers observed at height without personal fall arrest systems engaged. Site safety coordinator absent at time of inspection.,High,71.4
2024-09-23,Field Visit,Consultation,Open,WP07816184,"Montgomery, Hensley and Ray","4235 Christopher Court Suite 594, Burlington, ON",L1R 1X4,212110,Bituminous Coal Underground Mining,Employer,ORG103413,James Group,Requirement Time Based,Rescinded,Building Opportunities in the Skilled Trades Act,REG_632,Workplace Hazardous Materials Information System,26,(1),(f),Employer's health and safety policy was three years out of date and did not reference current regulatory requirements. No recent injuries on record at this site. Owner was cooperative and began policy review during inspection. Time-based order to update and repost policy within 21 days.,Low,20.6
2025-09-06,Offsite Visit,Consultation,In Progress,WP83503056,Jones-Gentry,"55341 Amanda Gardens Apt. 764, Oakville, ON",L5H 5P3,331110,Iron and Steel Mills,Worker,ORG953767,Charles Mcgee,Requirement Forthwith,Outstanding,Occupational Health and Safety Act,REG_490,Designated Substance - Asbestos on Construction Projects,55,,(b),Fire extinguisher inspection tags were 2 months overdue on 3 units in the facility. Extinguishers were in good physical condition with no visible damage or low pressure indicators. Forthwith order issued. Employer agreed to schedule a certified technician inspection on the same day of visit.,Low,22.5
2025-06-27,Field Visit,Inspection,In Progress,WP69166978,Maddox-Valencia,"653 William Course Apt. 122, Mississauga, ON",L5R 4C3,321111,Sawmills,Worker,ORG018451,Graham-Chavez,Time Unknown Order,Outstanding,Building Opportunities in the Skilled Trades Act,REG_388,Electrical Utility Safety Rules,27,(3),(b),Workers found operating forklifts without valid operator certification. Multiple near-miss incidents reported in the previous 30 days. Pedestrian zones not delineated and aisle markings absent. Inadequate lighting in storage areas creating collision risk. Safe work procedures for powered industrial trucks not available at workplace.,High,87.9
2026-04-27,Field Visit Support Role,Inspection,In Progress,WP88095701,Koch-Decker,"281 Skinner Parkways Apt. 252, Markham, ON",L5K 2W7,811111,General Automotive Repair,Supervisor,ORG430391,Dennis Inc,Forthwith Order,Complied With,Occupational Health and Safety Act,REG_490,Designated Substance - Asbestos on Construction Projects,50,(5),(a),Restaurant kitchen workers handling hot liquids without appropriate heat-resistant PPE. No written safe work procedure for handling cooking with hot oil. One thermal burn injury reported to management in past quarter. Compliance order requiring written procedure and training update issued.,Medium,22.9
2026-05-14,Offsite Visit,Inspection,Open,WP34657871,Walker LLC,"78248 Brandt Plains, Burlington, ON",L4N 1M6,238110,Poured Concrete Foundation Contractors,Supplier,ORG098393,Jones Ltd,Time Based Order,In-Process,Building Opportunities in the Skilled Trades Act,REG_420,Notices and Reports - Fatalities and Critical Injuries,27,(5),(a),Routine proactive inspection identified emergency exit signage requiring replacement


Total records generated : 100

Severity distribution (expected ≈ 10% Critical, 25% High, 40% Medium, 25% Low):
SEVERITY_LEVEL
Medium      35
High        29
Low         27
Critical     9
Name: count, dtype: int64

Risk score statistics (expected range: 0–100, mean ≈ 45):
count    100.00
mean      52.49
std       28.21
min        0.00
25%       27.78
50%       49.30
75%       75.52
max      100.00
Name: RISK_SCORE, dtype: float64

Order type distribution:
ORDER_TYPE
Stop Work Order           30
Requirement Forthwith     13
Requirement Time Based    12
Time Unknown Order        12
Plan Order                12
Time Based Order          11
Forthwith Order           10
Name: count, dtype: int64

✓  Saved → workspace.ohs_data.synthetic_inspections
    Go to Catalog → workspace → ohs_data → synthetic_inspections to verify.
